In [1]:
!pip install -q accelerate -U
!pip install -q bitsandbytes -U
!pip install -q trl -U
!pip install -q peft -U
!pip install -q transformers -U
!pip install -q datasets -U

In [2]:
!pip install torchinfo

In [3]:
from torchinfo import summary
import os
import pandas as pd
import torch
from datasets import load_dataset, Dataset, DatasetDict
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM
from trl.extras.dataset_formatting import FORMAT_MAPPING, instructions_formatting_function, conversations_formatting_function

In [4]:
df = pd.read_csv('https://github.com/laxmimerit/All-CSV-ML-Data-Files-Download/raw/master/amazon_product_details.csv', usecols=['category', 'about_product', 'product_name'])

In [5]:
df['category'] = df['category'].apply(lambda x: x.split('|')[-1])

In [6]:
products = df[['category', 'product_name']]
description = df[['category', 'about_product']]

products = products.rename(columns={'product_name': 'text'})
description = description.rename(columns={'about_product': 'text'})

products['task_type'] = 'Product Name'
description['task_type'] = 'Product Description'

In [7]:
df = pd.concat([products, description], ignore_index=True)

In [8]:
dataset = Dataset.from_pandas(df)
dataset = dataset.shuffle(seed=0)
dataset = dataset.train_test_split(test_size=0.1)

In [9]:
dataset

DatasetDict({
    train: Dataset({
        features: ['category', 'text', 'task_type'],
        num_rows: 2637
    })
    test: Dataset({
        features: ['category', 'text', 'task_type'],
        num_rows: 293
    })
})

In [10]:
dataset['test'][2], dataset['train'][2]

({'category': 'CompositionNotebooks',
  'text': 'Twin wiro binding|Paper color: White|Paper density: 70 gsm|No of pages 300',
  'task_type': 'Product Description'},
 {'category': 'SmartWatches',
  'text': 'The brilliant 1.3" colour display is now full capacitive touch, supporting taps and swipes, so it is easy to read and operate.|The strong polycarbonate case makes the ColorFit Pro 2 featherlight on your wrist and is available in 4 beautiful colours with matching swappable straps.|24x7 heart rate monitoring with the built in optical HR monitor that measures your heart rate every five minutes. With up to ten day battery life, ColorFit Pro 2 can go for more than a week without needing to be charged via the included magnetic charger.|9 sports modes to cover all your activities, whether you walk, run, hike, bike, treadmill, work-out, climb, spin, of perform yoga.|You can sweat as much as you like and even wear the ColorFit Pro 2 in the rain, thanks to its IP68 waterproof rating.',
  'task

In [11]:
repo_id = 'microsoft/Phi-3-mini-4k-instruct'
tokenizer_1 = AutoTokenizer.from_pretrained(repo_id)

base_model_id = "microsoft/phi-2"
model = AutoModelForCausalLM.from_pretrained(base_model_id, trust_remote_code=True,
                                             torch_dtype=torch.float16, load_in_8bit=True)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [12]:
tokenizer_1

LlamaTokenizerFast(name_or_path='microsoft/Phi-3-mini-4k-instruct', vocab_size=32000, model_max_length=4096, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '<|endoftext|>', 'unk_token': '<unk>', 'pad_token': '<|endoftext|>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=False),
	32000: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32001: AddedToken("<|assistant|>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=True),
	32002: AddedToken("<|placeholder1|>", rstrip=True, lstrip=False, single_word=False, normalized=False, special=Tr

In [13]:
tokenizer

CodeGenTokenizerFast(name_or_path='microsoft/phi-2', vocab_size=50257, model_max_length=2048, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50257: AddedToken("                               ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50258: AddedToken("                              ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50259: AddedToken("                             ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50260: AddedToken("                            ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50261: AddedToken("              

In [14]:
print(tokenizer.chat_template)
print(tokenizer_1.chat_template)

None
{% for message in messages %}{% if message['role'] == 'system' %}{{'<|system|>
' + message['content'] + '<|end|>
'}}{% elif message['role'] == 'user' %}{{'<|user|>
' + message['content'] + '<|end|>
'}}{% elif message['role'] == 'assistant' %}{{'<|assistant|>
' + message['content'] + '<|end|>
'}}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '<|assistant|>
' }}{% else %}{{ eos_token }}{% endif %}


In [15]:
tokenizer.chat_template = tokenizer_1.chat_template

In [16]:
print(tokenizer.chat_template)

{% for message in messages %}{% if message['role'] == 'system' %}{{'<|system|>
' + message['content'] + '<|end|>
'}}{% elif message['role'] == 'user' %}{{'<|user|>
' + message['content'] + '<|end|>
'}}{% elif message['role'] == 'assistant' %}{{'<|assistant|>
' + message['content'] + '<|end|>
'}}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '<|assistant|>
' }}{% else %}{{ eos_token }}{% endif %}


In [17]:
tokenizer.special_tokens_map

{'bos_token': '<|endoftext|>',
 'eos_token': '<|endoftext|>',
 'unk_token': '<|endoftext|>'}

In [18]:
special_tokens_dict = {'unk_token': '<unk>',
                       'eos_token': '<|endoftext|>',
                       'bos_token': '<s>',
                       'pad_token': '<pad>'
                       }
special_tokens_dict

{'unk_token': '<unk>',
 'eos_token': '<|endoftext|>',
 'bos_token': '<s>',
 'pad_token': '<pad>'}

In [19]:
tokenizer.add_special_tokens(special_tokens_dict)
tokenizer

CodeGenTokenizerFast(name_or_path='microsoft/phi-2', vocab_size=50257, model_max_length=2048, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '<|endoftext|>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50257: AddedToken("                               ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50258: AddedToken("                              ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50259: AddedToken("                             ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50260: AddedToken("                            ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50261: AddedToken("          

In [20]:
new_toks  = ['<|system|>', '<|user|>', '<|assistant|>', '<|end|>']
new_toks

['<|system|>', '<|user|>', '<|assistant|>', '<|end|>']

In [21]:
tokenizer.add_tokens(new_toks)

4

In [22]:
tokenizer

CodeGenTokenizerFast(name_or_path='microsoft/phi-2', vocab_size=50257, model_max_length=2048, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '<|endoftext|>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=True, added_tokens_decoder={
	50256: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50257: AddedToken("                               ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50258: AddedToken("                              ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50259: AddedToken("                             ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50260: AddedToken("                            ", rstrip=False, lstrip=False, single_word=False, normalized=True, special=False),
	50261: AddedToken("          

In [23]:
model.model.embed_tokens

Embedding(51200, 2560)

In [24]:
tokenizer.padding_side='left'
tokenizer.padding_side

'left'

In [25]:
def create_rol_asst(example):
  example = dict(example)
  messages = []

  for k in example.keys():
    #print("key: ", k)
    if k == 'category':
      #print("entered category")
      role = 'user'
      content = f'For ###category: {example["category"]}, create ###task_type: {example["task_type"]}'
      user_dict = {'role': role, 'content': content}
    elif k == 'text':
      #print("entered assistant")
      role = 'assistant'
      content = example['text']
      asst_dict = {'role': role, 'content': content}
    else:
      #print("entered pass")
      pass
  messages.append(user_dict)
  messages.append(asst_dict)
  return {'messages': messages}

In [26]:
train_ds = dataset['train']
test_ds = dataset['test']
train_ds, test_ds

(Dataset({
     features: ['category', 'text', 'task_type'],
     num_rows: 2637
 }),
 Dataset({
     features: ['category', 'text', 'task_type'],
     num_rows: 293
 }))

In [27]:
train_ds = train_ds.map(create_rol_asst, batched=False)

Map:   0%|          | 0/2637 [00:00<?, ? examples/s]

In [28]:
test_ds = test_ds.map(create_rol_asst, batched=False)

Map:   0%|          | 0/293 [00:00<?, ? examples/s]

In [29]:
train_ds, test_ds

(Dataset({
     features: ['category', 'text', 'task_type', 'messages'],
     num_rows: 2637
 }),
 Dataset({
     features: ['category', 'text', 'task_type', 'messages'],
     num_rows: 293
 }))

In [30]:
train_ds = train_ds.remove_columns(['category', 'text', 'task_type'])
test_ds = test_ds.remove_columns(['category', 'text', 'task_type'])
train_ds, test_ds

(Dataset({
     features: ['messages'],
     num_rows: 2637
 }),
 Dataset({
     features: ['messages'],
     num_rows: 293
 }))

In [31]:
FORMAT_MAPPING['chatml'] == train_ds.features['messages']

True

In [32]:
messages = train_ds["messages"][0]
output_texts = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
print(output_texts)

<|user|>
For ###category: USBCables, create ###task_type: Product Description<|end|>
<|assistant|>
[High Compatibility] : This iphone data cable supports with iPhone 6,6s,6 plus,6s plus,7 7 plus ,8 8plus,x,xs,11 pro max,12 mini pro max,13 mini pro max iPad Air, iPad mini, iPod Nano and iPod Touch|[Fast Charge&Data Sync ] : It can charge and sync simultaneously at a rapid speed, Compatible with any charging adaptor, multi-port charging station or power bank ,for fast charging ,fast adapter is must.|😍【Durable Spring Protection】：The easy-to-break connection port is protected by spring, which is a flexible and durable cable.You can use it with confidence.|【 Ultra High Quality】: According to the experimental results, the fishbone design can accept at least 20,000 bending and insertion tests for extra protection and durability. Upgraded 3D aluminum connector and exclusive laser welding technology, which to ensure the metal part won't break and also have a tighter connection which fits well e

In [33]:
messages = test_ds["messages"][10]
output_texts = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
print(output_texts)

<|user|>
For ###category: TripodLegs, create ###task_type: Product Name<|end|>
<|assistant|>
DIGITEK® (DTR-200MT) (18 CM) Portable & Flexible Mini Tripod with Mobile Holder & 360 Degree Ball Head, For Smart Phones, Compact Cameras, GoPro, Maximum Operating Height: 7.87 Inch, Maximum Load Upto: 1 kgs<|end|>
<|endoftext|>


In [34]:
summary(model)

Layer (type:depth-idx)                             Param #
PhiForCausalLM                                     --
├─PhiModel: 1-1                                    --
│    └─Embedding: 2-1                              131,072,000
│    └─ModuleList: 2-2                             --
│    │    └─PhiDecoderLayer: 3-1                   78,671,360
│    │    └─PhiDecoderLayer: 3-2                   78,671,360
│    │    └─PhiDecoderLayer: 3-3                   78,671,360
│    │    └─PhiDecoderLayer: 3-4                   78,671,360
│    │    └─PhiDecoderLayer: 3-5                   78,671,360
│    │    └─PhiDecoderLayer: 3-6                   78,671,360
│    │    └─PhiDecoderLayer: 3-7                   78,671,360
│    │    └─PhiDecoderLayer: 3-8                   78,671,360
│    │    └─PhiDecoderLayer: 3-9                   78,671,360
│    │    └─PhiDecoderLayer: 3-10                  78,671,360
│    │    └─PhiDecoderLayer: 3-11                  78,671,360
│    │    └─PhiDecoderLayer: 3-12 

In [35]:
messages = test_ds["messages"][10]
output_texts = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
print(output_texts)

<|user|>
For ###category: TripodLegs, create ###task_type: Product Name<|end|>
<|assistant|>



In [36]:
# tokenize -> generate -> decode
max_length = 512
model_input = tokenizer(
      output_texts,
      truncation = True,
      max_length=max_length,
      padding = "max_length",
      return_tensors='pt'
  ).to("cuda")


In [37]:
model.eval()

with torch.no_grad():
  output = model.generate(**model_input, max_new_tokens=256,
                          repetition_penalty=1.15)
  result = tokenizer.decode(output[0], skip_special_tokens=True)

  print(result)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


<|user|>
For ###category: TripodLegs, create ###task_type: Product Name<|end|>
<|assistant|>
###description: This is a description of the product.
###price: $$$
###image: /path/to/product-image.jpg



In [38]:
del model

In [39]:
base_model_id = "microsoft/phi-2"
model = AutoModelForCausalLM.from_pretrained(base_model_id, trust_remote_code=True,
                                             torch_dtype=torch.float16, load_in_8bit=True)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [40]:
model

PhiForCausalLM(
  (model): PhiModel(
    (embed_tokens): Embedding(51200, 2560)
    (layers): ModuleList(
      (0-31): 32 x PhiDecoderLayer(
        (self_attn): PhiAttention(
          (q_proj): Linear8bitLt(in_features=2560, out_features=2560, bias=True)
          (k_proj): Linear8bitLt(in_features=2560, out_features=2560, bias=True)
          (v_proj): Linear8bitLt(in_features=2560, out_features=2560, bias=True)
          (dense): Linear8bitLt(in_features=2560, out_features=2560, bias=True)
        )
        (mlp): PhiMLP(
          (activation_fn): NewGELUActivation()
          (fc1): Linear8bitLt(in_features=2560, out_features=10240, bias=True)
          (fc2): Linear8bitLt(in_features=10240, out_features=2560, bias=True)
        )
        (input_layernorm): LayerNorm((2560,), eps=1e-05, elementwise_affine=True)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (rotary_emb): PhiRotaryEmbedding()
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (final_

In [41]:
model = prepare_model_for_kbit_training(model)
summary(model)

Layer (type:depth-idx)                             Param #
PhiForCausalLM                                     --
├─PhiModel: 1-1                                    --
│    └─Embedding: 2-1                              (131,072,000)
│    └─ModuleList: 2-2                             --
│    │    └─PhiDecoderLayer: 3-1                   (78,671,360)
│    │    └─PhiDecoderLayer: 3-2                   (78,671,360)
│    │    └─PhiDecoderLayer: 3-3                   (78,671,360)
│    │    └─PhiDecoderLayer: 3-4                   (78,671,360)
│    │    └─PhiDecoderLayer: 3-5                   (78,671,360)
│    │    └─PhiDecoderLayer: 3-6                   (78,671,360)
│    │    └─PhiDecoderLayer: 3-7                   (78,671,360)
│    │    └─PhiDecoderLayer: 3-8                   (78,671,360)
│    │    └─PhiDecoderLayer: 3-9                   (78,671,360)
│    │    └─PhiDecoderLayer: 3-10                  (78,671,360)
│    │    └─PhiDecoderLayer: 3-11                  (78,671,360)
│    │    

(q_proj): Linear8bitLt(in_features=2560, out_features=2560, bias=True)
          (k_proj): Linear8bitLt(in_features=2560, out_features=2560, bias=True)
          (v_proj): Linear8bitLt(in_features=2560, out_features=2560, bias=True)
          (dense): Linear8bitLt(in_features=2560, out_features=2560, bias=True)

In [42]:
target_modules = ["q_proj", "k_proj", "v_proj", "dense",  "fc1", "fc2"]

config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules = target_modules,
    bias = "none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

In [43]:
summary(model)

Layer (type:depth-idx)                                            Param #
PeftModelForCausalLM                                              --
├─LoraModel: 1-1                                                  --
│    └─PhiForCausalLM: 2-1                                        --
│    │    └─PhiModel: 3-1                                         2,695,746,560
│    │    └─Linear: 3-2                                           (131,123,200)
Total params: 2,826,869,760
Trainable params: 47,185,920
Non-trainable params: 2,779,683,840

In [44]:
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): PhiForCausalLM(
      (model): PhiModel(
        (embed_tokens): Embedding(51200, 2560)
        (layers): ModuleList(
          (0-31): 32 x PhiDecoderLayer(
            (self_attn): PhiAttention(
              (q_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=2560, out_features=2560, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2560, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2560, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Line

In [45]:
trainable_parms, tot_parms = model.get_nb_trainable_parameters()
print(f'Trainable parameters:             {trainable_parms/1e6:.2f}M')
print(f'Total parameters:                 {tot_parms/1e6:.2f}M')
print(f'Fraction of trainable parameters: {100*trainable_parms/tot_parms:.2f}%')

Trainable parameters:             47.19M
Total parameters:                 2826.87M
Fraction of trainable parameters: 1.67%


In [46]:
print(model.get_memory_footprint()/1e6)

3757.731904


In [47]:
sft_config = SFTConfig(
    ## GROUP 1: Memory usage
    # These arguments will squeeze the most out of your GPU's RAM
    # Checkpointing
    #gradient_checkpointing=True,
    # this saves a LOT of memory
    # Set this to avoid exceptions in newer versions of PyTorch
    #gradient_checkpointing_kwargs={'use_reentrant': False},
    # Gradient Accumulation / Batch size
    # Actual batch (for updating) is same (1x) as micro-batch size
    gradient_accumulation_steps=1,
    # The initial (micro) batch size to start off with
    per_device_train_batch_size=128,
    # If batch size would cause OOM, halves its size until it works
    auto_find_batch_size=True,

    ## GROUP 2: Dataset-related
    max_seq_length=128,
    # Dataset
    # packing a dataset means no padding is needed
    packing=False,

    ## GROUP 3: These are typical training parameters
    num_train_epochs=10,
    learning_rate=1e-3,
    # Optimizer
    # 8-bit Adam optimizer - doesn't help much if you're using LoRA!
    optim='adamw_torch',

    ## GROUP 4: Logging parameters
    #logging_steps=50,
    logging_dir='./logs',
    output_dir='./phi2-chat-adapter',
    report_to='none',

    eval_strategy="epoch", # Evaluate the model every logging step
    #eval_steps=25,               # Evaluate and save checkpoints every 50 steps
    do_eval=True,                # Perform evaluation at the end of training
)

In [48]:
tokenizer.padding_side='left'
response_template = '<|assistant|>' # according to the tokenizer's chat template
collator_fn=DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

In [49]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=collator_fn
    )

Converting train dataset to ChatML:   0%|          | 0/2637 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/2637 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2637 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2637 [00:00<?, ? examples/s]

Converting eval dataset to ChatML:   0%|          | 0/293 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/293 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/293 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/293 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [50]:
dl = trainer.get_train_dataloader()
batch = next(iter(dl))
batch['input_ids'][0], batch['labels'][0]

(tensor([50299,   198,  1890, 44386, 22872,    25,  8765,  5248,  3979,    11,
          2251, 44386, 35943,    62,  4906,    25,  8721, 12489, 50301,   198,
         50300,   198,    39, 18060,   347, 10705, 23336,    54,   350,    13,
            44,    13,    47,    13,    46,    12,  3242,   364, 10758, 36220,
          5072,    13, 16386,  6462,  5572,  2128,   351, 10862,    88, 12702,
           351,   663,   734,  3665, 14729,   286, 11636,    91,    44, 16724,
          4061,  2538,  7102, 48842,  3824,  9050,    12,  2080,   663,  8036,
          2493,  1799,  1222, 19843,    11,   345,   460,  2018,   284,   428,
         10834,  2884, 19263,    11,  8450,    11, 18695,  8829,    11, 27548,
            55, 20249,  1222,  7631,    91,  1340,    12,  8202,  9795, 21728,
         11357, 49833,    50,    12,   632,   468,  1550,    12, 29828, 20969,
         36357,   351,   477,   286,   262, 11244,   345,   761,    13,   921,
           460,  3811,    14, 49991,  7849,    11, 2

In [53]:
len(batch['input_ids'][0])

128

In [ ]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/utils/checkpoint.py:460: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:315: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss
1,No log,2.654436
2,No log,2.392817
3,No log,2.342238
4,No log,2.349379
5,No log,2.481648
6,No log,2.569603
7,1.477600,2.684989
8,1.477600,2.785347
9,1.477600,2.833122
10,1.477600,2.891157


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/torch/utils/checkpoint.py:460: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:315: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:315: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quanti

TrainOutput(global_step=830, training_loss=0.9823007422757436, metrics={'train_runtime': 9586.5913, 'train_samples_per_second': 2.751, 'train_steps_per_second': 0.087, 'total_flos': 5.472823141466112e+16, 'train_loss': 0.9823007422757436})

In [76]:
def gen_output(messages):
  output_texts = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
  print(output_texts)

  # tokenize -> generate -> decode
  max_length = 512
  model_input = tokenizer(
      output_texts,
      truncation = True,
      max_length=max_length,
      padding = "max_length",
      return_tensors='pt'
  ).to("cuda")

  model.eval()
  with torch.no_grad():
    output = model.generate(**model_input, max_new_tokens=128,
                            repetition_penalty=1.15,
                            eos_token_id=tokenizer.eos_token_id,
                            pad_token_id=tokenizer.pad_token_id,
                            num_beams=4,
                            top_k=3,
                            do_sample=True
                            )
  result = tokenizer.decode(output[0], skip_special_tokens=True)
  print(result)

In [77]:
messages = test_ds["messages"][10]
gen_output(messages)

<|user|>
For ###category: TripodLegs, create ###task_type: Product Name<|end|>
<|assistant|>

<|user|>
For ###category: TripodLegs, create ###task_type: Product Name<|end|>
<|assistant|>
WeCool Bluetooth Extendable Selfie Sticks with Wireless Remote and Tripod Stand, 3-in-1 Multifunctional Selfie Stick with Tripod Stand Compatible with iPhone/OnePlus/Samsung/Oppo/Vivo and All Phones サーティワン



In [78]:
messages = test_ds["messages"][15]
gen_output(messages)

<|user|>
For ###category: USBCables, create ###task_type: Product Name<|end|>
<|assistant|>

<|user|>
For ###category: USBCables, create ###task_type: Product Name<|end|>
<|assistant|>
Wayona Usb Type C Fast Charger Cable Fast Charging Usb C Cable/Cord Compatible For Samsung Galaxy S10E S10 S9 S8 Plus S10+,Note 10 Note 9 Note 8,S20,M31S,M40,Realme X3,Pixel 2 Xl (3 Ft Pack Of 1,Grey) UCHIJ



In [64]:
model.config.pad_token_id = tokenizer.pad_token_id

In [79]:
messages = test_ds["messages"][25]
gen_output(messages)

<|user|>
For ###category: Rice&PastaCookers, create ###task_type: Product Description<|end|>
<|assistant|>

<|user|>
For ###category: Rice&PastaCookers, create ###task_type: Product Description<|end|>
<|assistant|>
Multi functional use- Steamed veggies and rice, various recipes you can accomplish it all with your rice cooker|Cooks up to 600 gms of raw rice or up to 5.5 cups of raw rice|Multi cooking functions-White rice, brown rice with low carb, short grain, porridge, steam, slow cook|ADVANCED FUZZY LOGIC RICE COOKER TECHNOLOGY - auto adjust temperature and timings for optimal rice cooking combined with tailored useful add on options like steamed veggies, fried rice, porridge, etc|Cooks up to 600 gms of raw rice or up


In [80]:
messages = test_ds["messages"][40]
gen_output(messages)

<|user|>
For ###category: EggBoilers, create ###task_type: Product Name<|end|>
<|assistant|>

<|user|>
For ###category: EggBoilers, create ###task_type: Product Name<|end|>
<|assistant|>
Simxen Egg Boiler Electric Automatic Off 7 Egg Poacher for Steaming, Cooking Also Boiling and Frying 400 W (Blue, Pink) UCHIJ



In [83]:
messages = test_ds["messages"][2]
gen_output(messages)

<|user|>
For ###category: CompositionNotebooks, create ###task_type: Product Description<|end|>
<|assistant|>

<|user|>
For ###category: CompositionNotebooks, create ###task_type: Product Description<|end|>
<|assistant|>
The cover design of the notebook is subject to change, it depends on stock availability|Long Notebook - 140 Pages, Single Line, 297mm x 210mm (Pack of 12)|Notebooks for every subject for hassle-free note-taking during classes or lectures.|Classmate uses and elemental chlorine free paper|This notebook consists of papers UCHIJ



In [84]:
messages = test_ds["messages"][42]
gen_output(messages)

<|user|>
For ###category: SmartWatches, create ###task_type: Product Name<|end|>
<|assistant|>

<|user|>
For ###category: SmartWatches, create ###task_type: Product Name<|end|>
<|assistant|>
boAt Xtend Smartwatch with Alexa Built-in, 1.69” HD Display, Multiple Watch Faces, Stress Monitor, Heart & SpO2 Monitoring, 14 Sports Modes, Sleep Monitor, 5 ATM & 7 Days Battery(Charcoal Black) UCHIJ



In [85]:
trainer.save_model('phi_2_chat_non_chat_prod_descr_name')

In [86]:
from huggingface_hub import login
login()

In [88]:
trainer.push_to_hub()

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/189M [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.56k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Fardan/phi2-chat-adapter/commit/a303fb10b1e2e6a6f8022f2a72b222e9d3bf4635', commit_message='End of training', commit_description='', oid='a303fb10b1e2e6a6f8022f2a72b222e9d3bf4635', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Fardan/phi2-chat-adapter', endpoint='https://huggingface.co', repo_type='model', repo_id='Fardan/phi2-chat-adapter'), pr_revision=None, pr_num=None)